In [2]:
# cell 1 — imports and connection
import pandas as pd
import sqlite3
from pathlib import Path

DB_PATH = Path("../data/database/steam.db")
conn = sqlite3.connect(DB_PATH)

print("Connected to:", DB_PATH)
print("Tables available:")
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

Connected to: ..\data\database\steam.db
Tables available:


,name
0,games
1,game_platforms
2,game_genres
3,game_tags


In [3]:
# cell 2 — genre performance ranking
query = """
SELECT 
    g.genre,
    COUNT(DISTINCT gm.appid)         AS total_games,
    ROUND(AVG(gm.owner_midpoint), 0) AS avg_owners,
    ROUND(AVG(gm.positive_ratio), 3) AS avg_positive_ratio,
    ROUND(AVG(gm.success_score), 2)  AS avg_success_score,
    ROUND(AVG(gm.price), 2)          AS avg_price
FROM game_genres g
JOIN games gm ON g.appid = gm.appid
WHERE gm.total_ratings > 10
AND g.genre NOT IN (
    'Early Access', 'Free to Play', 'Indie',
    'Gore', 'Violent', 'Nudity', 'Sexual Content',
    'Animation & Modeling', 'Design & Illustration',
    'Utilities', 'Audio Production', 'Video Production',
    'Web Publishing', 'Education', 'Software Training'
)
GROUP BY g.genre
HAVING total_games > 50
ORDER BY avg_success_score DESC
"""

genre_performance = pd.read_sql(query, conn)
print(genre_performance.to_string())

                   genre  total_games  avg_owners  avg_positive_ratio  avg_success_score  avg_price
0  Massively Multiplayer          631    705753.0               0.625               8.67       5.19
1                    RPG         3417    201288.0               0.733               7.87       9.49
2                 Action         8811    264472.0               0.723               7.79       8.53
3               Strategy         4053    186493.0               0.705               7.76       9.40
4              Adventure         7592    141864.0               0.732               7.59       8.38
5                 Racing          760    109980.0               0.686               7.54      10.09
6             Simulation         4015    128041.0               0.682               7.47       9.76
7                 Sports          919    108776.0               0.700               7.34      11.30
8                 Casual         6835     74954.0               0.723               7.33       5.33


In [4]:
# cell 3 — save genre performance query
with open("../sql/01_genre_performance.sql", "w") as f:
    f.write(query)

genre_performance.to_csv("../data/processed/genre_performance.csv", index=False)
print("Saved SQL and CSV")

Saved SQL and CSV


In [5]:
# cell 4 — price tier analysis
query2 = """
SELECT
    CASE
        WHEN price = 0                THEN '1. Free'
        WHEN price < 5                THEN '2. Under $5'
        WHEN price BETWEEN 5 AND 10   THEN '3. $5 - $10'
        WHEN price BETWEEN 10 AND 20  THEN '4. $10 - $20'
        WHEN price BETWEEN 20 AND 40  THEN '5. $20 - $40'
        ELSE                               '6. Over $40'
    END AS price_tier,
    COUNT(*)                             AS total_games,
    ROUND(AVG(owner_midpoint), 0)        AS avg_owners,
    ROUND(AVG(positive_ratio), 3)        AS avg_positive_ratio,
    ROUND(AVG(success_score), 2)         AS avg_success_score,
    ROUND(AVG(average_playtime), 1)      AS avg_playtime_mins
FROM games
WHERE total_ratings > 10
GROUP BY price_tier
ORDER BY price_tier
"""

price_tiers = pd.read_sql(query2, conn)
print(price_tiers.to_string())

     price_tier  total_games  avg_owners  avg_positive_ratio  avg_success_score  avg_playtime_mins
0       1. Free         2272    497027.0               0.718               8.14              521.8
1   2. Under $5         5386     55781.0               0.683               7.35               90.8
2   3. $5 - $10         7049    111023.0               0.738               7.42               85.8
3  4. $10 - $20         3954    191220.0               0.770               7.82              208.8
4  5. $20 - $40         1094    432221.0               0.764               8.46              506.9
5   6. Over $40          269    436190.0               0.719               9.02             1437.8


In [6]:
# cell 5 — save price tier
with open("../sql/02_price_tier_analysis.sql", "w") as f:
    f.write(query2)

price_tiers.to_csv("../data/processed/price_tiers.csv", index=False)
print("Saved")

Saved


In [7]:
# cell 6 — top 3 games per genre (window function)
query3 = """
WITH ranked_games AS (
    SELECT
        gm.name,
        gm.owner_midpoint,
        gm.positive_ratio,
        gm.success_score,
        gm.price,
        g.genre,
        RANK() OVER (
            PARTITION BY g.genre
            ORDER BY gm.success_score DESC
        ) AS genre_rank
    FROM games gm
    JOIN game_genres g ON gm.appid = g.appid
    WHERE gm.total_ratings > 50
    AND g.genre NOT IN (
        'Early Access', 'Free to Play', 'Indie',
        'Gore', 'Violent', 'Nudity', 'Sexual Content',
        'Animation & Modeling', 'Design & Illustration',
        'Utilities', 'Audio Production', 'Video Production',
        'Web Publishing', 'Education', 'Software Training'
    )
)
SELECT *
FROM ranked_games
WHERE genre_rank <= 3
ORDER BY genre, genre_rank
"""

top_games = pd.read_sql(query3, conn)
print(top_games.to_string())

                                name  owner_midpoint  positive_ratio  success_score  price                  genre  genre_rank
0                             Dota 2       150000000        0.858710      16.203357   0.00                 Action           1
1      PLAYERUNKNOWN'S BATTLEGROUNDS        75000000        0.504632      15.705277  34.28                 Action           2
2   Counter-Strike: Global Offensive        75000000        0.867952      15.699414   0.00                 Action           3
3      PLAYERUNKNOWN'S BATTLEGROUNDS        75000000        0.504632      15.705277  34.28              Adventure           1
4                           Unturned        35000000        0.902850      14.585432   0.00              Adventure           2
5                 Grand Theft Auto V        15000000        0.702568      14.324695  31.74              Adventure           3
6                           Unturned        35000000        0.902850      14.585432   0.00                 Casual     

In [8]:
# cell 7 — save top games per genre
with open("../sql/03_top_games_per_genre.sql", "w") as f:
    f.write(query3)

top_games.to_csv("../data/processed/top_games_per_genre.csv", index=False)
print("Saved")

Saved


In [9]:
# cell 8 — tag rankings by avg owners
query4 = """
SELECT
    t.tag,
    COUNT(DISTINCT gm.appid)         AS total_games,
    ROUND(AVG(gm.owner_midpoint), 0) AS avg_owners,
    ROUND(AVG(gm.positive_ratio), 3) AS avg_positive_ratio,
    ROUND(AVG(gm.success_score), 2)  AS avg_success_score,
    ROUND(AVG(gm.price), 2)          AS avg_price
FROM game_tags t
JOIN games gm ON t.appid = gm.appid
WHERE gm.total_ratings > 10
GROUP BY t.tag
HAVING total_games > 100
ORDER BY avg_success_score DESC
LIMIT 20
"""

tag_rankings = pd.read_sql(query4, conn)
print(tag_rankings.to_string())

                 tag  total_games  avg_owners  avg_positive_ratio  avg_success_score  avg_price
0         Open World          240   1162604.0               0.725              10.63      20.97
1        Multiplayer          394   1981612.0               0.735              10.23      11.45
2              Co-op          111   1748198.0               0.816              10.09      14.19
3                FPS          391   1548645.0               0.732               9.62      10.76
4            Fantasy          107    561916.0               0.755               9.40      14.99
5            Zombies          148    986047.0               0.725               9.39      10.39
6           Survival          225   1187933.0               0.679               9.34      14.10
7         Story Rich          148    558243.0               0.861               9.26      15.77
8             Sci-fi          150    724833.0               0.760               9.25      12.71
9                RTS          168    475

In [10]:
# cell 9 — save tag rankings
with open("../sql/03_tag_rankings.sql", "w") as f:
    f.write(query4)

tag_rankings.to_csv("../data/processed/tag_rankings.csv", index=False)
print("Saved")

Saved


In [11]:
# cell 10 — developer comparison (indie vs prolific studios)
query5 = """
WITH developer_stats AS (
    SELECT
        developer,
        COUNT(*)                             AS total_games,
        ROUND(AVG(owner_midpoint), 0)        AS avg_owners,
        ROUND(AVG(positive_ratio), 3)        AS avg_positive_ratio,
        ROUND(AVG(success_score), 2)         AS avg_success_score,
        ROUND(AVG(price), 2)                 AS avg_price,
        SUM(owner_midpoint)                  AS total_owners
    FROM games
    WHERE total_ratings > 10
    GROUP BY developer
    HAVING total_games >= 1
),
developer_tier AS (
    SELECT *,
        CASE
            WHEN total_games = 1      THEN '1. Single Title'
            WHEN total_games BETWEEN 2 AND 5   THEN '2. Small Studio (2-5)'
            WHEN total_games BETWEEN 6 AND 20  THEN '3. Mid Studio (6-20)'
            ELSE                               '4. Large Studio (20+)'
        END AS studio_tier
    FROM developer_stats
)
SELECT
    studio_tier,
    COUNT(*)                             AS total_developers,
    ROUND(AVG(total_games), 1)           AS avg_games_made,
    ROUND(AVG(avg_owners), 0)            AS avg_owners,
    ROUND(AVG(avg_positive_ratio), 3)    AS avg_positive_ratio,
    ROUND(AVG(avg_success_score), 2)     AS avg_success_score,
    ROUND(AVG(avg_price), 2)             AS avg_price
FROM developer_tier
GROUP BY studio_tier
ORDER BY studio_tier
"""

developer_comparison = pd.read_sql(query5, conn)
print(developer_comparison.to_string())

             studio_tier  total_developers  avg_games_made  avg_owners  avg_positive_ratio  avg_success_score  avg_price
0        1. Single Title              9800             1.0    133989.0               0.735               7.39       7.99
1  2. Small Studio (2-5)              2582             2.6    172940.0               0.733               7.78       8.63
2   3. Mid Studio (6-20)               305             9.1    226826.0               0.705               8.19       8.83
3  4. Large Studio (20+)                23            30.3    731322.0               0.709               7.57       9.34


In [12]:
# cell 11 — save developer comparison
with open("../sql/04_developer_comparison.sql", "w") as f:
    f.write(query5)

developer_comparison.to_csv("../data/processed/developer_comparison.csv", index=False)
print("Saved")

Saved
